# Palmer Penguins: ML Exploration (PCA & Clustering)

In this notebook, I'm going to explore the Palmer Penguins dataset using some machine learning techniques. The goal is to see if we can use physical measurements (like bill length and body mass) to group the penguins naturally without looking at their species labels first. 

I'll be using **Principal Component Analysis (PCA)** to simplify the data dimensions and **K-Means Clustering** to find the groups.

### 1. Setup and Imports

I'm using pandas for data handling, scikit-learn for the ML parts, and Plotly for the interactive charts.

In [55]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.express as px
import plotly.graph_objects as go
from preprocess import load_clean_data

# Defining the features I'll use for numerical analysis
ML_FEATURES = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
]

### 2. Loading the Dataset

I'll load the cleaned data and take a quick look to make sure everything looks right.

In [56]:
df = load_clean_data()
print(f"Loaded {len(df)} rows.")
df[ML_FEATURES].head()

Loaded 333 rows.


,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,39.1,18.7,181.0,3750.0
1,39.5,17.4,186.0,3800.0
2,40.3,18.0,195.0,3250.0
3,36.7,19.3,193.0,3450.0
4,39.3,20.6,190.0,3650.0


### 3. Feature Scaling

Since things like body mass are in grams (thousands) and bill length is in mm (tens), I need to scale them first. Otherwise, the body mass would dominate the PCA just because the numbers are bigger.

In [57]:
# Drop rows with missing feature values before scaling
df_ml = df.dropna(subset=ML_FEATURES).copy()
X = df_ml[ML_FEATURES].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Data scaled and ready for PCA.")

Data scaled and ready for PCA.


### 4. PCA: 2D vs 3D Comparison

I want to see how much information (variance) we keep when we reduce these 4 features down to 2 or 3 components.

In [58]:
# 2-Component PCA
pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_scaled)
var2 = pca2.explained_variance_ratio_.sum() * 100

# 3-Component PCA
pca3 = PCA(n_components=3, random_state=42)
X_pca3 = pca3.fit_transform(X_scaled)
var3 = pca3.explained_variance_ratio_.sum() * 100

print(f"2-Component PCA explains {var2:.1f}% of the variance.")
print(f"3-Component PCA explains {var3:.1f}% of the variance.")
print(f"PC3 adds another {var3-var2:.1f}% to the total.")

2-Component PCA explains 88.1% of the variance.
3-Component PCA explains 97.3% of the variance.
PC3 adds another 9.2% to the total.


### 5. K-Means Clustering

Since there are 3 main penguin species in this data, I'll try to find 3 clusters.

In [59]:
# Run clustering on the scaled data (the same space PCA uses)
k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df_ml["cluster"] = [f"Cluster {i}" for i in kmeans.fit_predict(X_scaled)]

# Add PCA coordinates for plotting
df_ml["PC1"] = X_pca2[:, 0]
df_ml["PC2"] = X_pca2[:, 1]
df_ml["PC3"] = X_pca3[:, 2]

# Check how well separated they are
score = silhouette_score(X_scaled, df_ml["cluster"])
print(f"Silhouette Score: {score:.3f}")

Silhouette Score: 0.446


### 6. Visualizing the Clusters (2D & 3D)

Let's see if our clusters actually match up with the different species. I'll look at both the 2D view and a 3D view to see if the extra dimension helps us see the gaps better.

In [60]:
# 1. 2D PCA Scatter
fig_2d = px.scatter(
    df_ml, x='PC1', y='PC2', 
    color='cluster', 
    symbol='species',
    title='2D PCA Space: Cluster vs Species',
    labels={'PC1': 'Principal Component 1', 'PC2': 'Principal Component 2'},
    width=800, height=600,
    template='plotly_white'
)
fig_2d.update_layout(title_x=0.5)
fig_2d.show()

# 2. 3D PCA Scatter
fig_3d = px.scatter_3d(
    df_ml, x='PC1', y='PC2', z='PC3',
    color='cluster', 
    symbol='species',
    title='3D PCA Space: Exploring the Extra Dimension',
    labels={'PC1': 'PC1', 'PC2': 'PC2', 'PC3': 'PC3'},
    width=900, height=700,
    template='plotly_white'  # Dark theme often looks better for 3D
)
# Make the markers larger and cleaner
fig_3d.update_traces(marker=dict(size=5, opacity=0.8, line=dict(width=1, color='DarkSlateGrey')))
fig_3d.update_layout(
    scene_camera=dict(eye=dict(x=1.8, y=1.8, z=0.8)),
    dragmode='orbit'
    title_x=0.5
)
fig_3d.show()


### 7. Characterizing the Clusters

Why do Cluster 0 and Cluster 2 overlap so much? Let's look at the average physical measurements for each cluster. This 'fingerprint' tells us what the K-Means algorithm actually found.

In [61]:
# Calculate means for each cluster
cluster_means = df_ml.groupby('cluster')[ML_FEATURES].mean()

# Normalize for the heatmap (so we can compare different units)
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

fig_heatmap = px.imshow(
    cluster_means_norm.T,
    labels=dict(x='Cluster', y='Feature', color='Relative Value'),
    title='Cluster Fingerprints: Average Feature Values',
    color_continuous_scale='Blues',
    aspect='auto'
)
fig_heatmap.update_layout(title_x=0.5)
fig_heatmap.show()


### 8. Interpretation: The "Adelie vs Chinstrap" Challenge

Looking at the plots and the heatmap, we can see a few things:
1. **Gentoo penguins (Cluster 1)** are very distinct. They have much larger bodies and longer flippers, making them easy for the algorithm to isolate.
2. **Adelie and Chinstrap penguins** are physically very similar in terms of body mass and flipper length. Their main difference is often the 'shape' of their bill (depth vs length), which is a subtle signal.
3. **Conclusion**: While K-Means finds 3 groups, the physical overlap between Adelie and Chinstrap means their clusters will never be perfectly separated in a 2D or 3D space based *only* on these 4 measurements. This is why the visualization looks 'messy' in that region — because the penguins themselves are 'messy' in nature!

### 7. Understanding PCA Loadings (Including PC3)

**Loadings** tell us which original measurements (bill, body mass, etc.) are most influential for each Principal Component. Let's see what the 3rd component actually captures.

In [62]:
loadings = pd.DataFrame(
    pca3.components_.T, index=ML_FEATURES, columns=["PC1", "PC2", "PC3"]
)

# Loadings bar chart
loadings_long = loadings.reset_index().melt(
    id_vars="index", var_name="Component", value_name="Loading"
)
fig_load = px.bar(
    loadings_long, x="index", y="Loading", color="Component",
    barmode="group", title="Feature Contribution to the First 3 Components"
)
fig_load.update_layout(template="plotly_white", title_x=0.5, xaxis_title="Feature")
fig_load.show()

# Print summary to be clear
print("Strongest driver for each component:")
print(f"PC1: {loadings['PC1'].abs().idxmax()} ({loadings['PC1'].abs().max():.2f})")
print(f"PC2: {loadings['PC2'].abs().idxmax()} ({loadings['PC2'].abs().max():.2f})")
print(f"PC3: {loadings['PC3'].abs().idxmax()} ({loadings['PC3'].abs().max():.2f})")

Strongest driver for each component:
PC1: flipper_length_mm (0.58)
PC2: bill_depth_mm (0.80)
PC3: bill_length_mm (0.64)


### 8. Final Interpretation

After looking at the results, here's what I've noticed:
* **Gentoo Penguins** are extremely distinct. They basically own their own cluster because they are much larger and have long, shallow bills.
* **Adelie and Chinstrap** penguins have a lot of overlap. Even in 3D, you can see them crowding together, which explains why the K-Means algorithm doesn't separate them perfectly.
* **PC3 Interpretation**: The 3rd component seems mostly driven by body mass and flipper length (depending on the exact variance). While it adds ~9% more variance, the clusters are already quite clear in 2D. The 3D view is cool for seeing the structure, but 2D is probably enough for the final dashboard.